# Evaluation Plots

Unified visualization notebook for comparing model performance across
architectures (Vanilla RNN, LSTM, MHA Vanilla, Trial-Context Hybrid MHA),
sessions (early / late), and hyperparameter grids.

All sweep-level plots are **faceted by session** (early vs late side-by-side).

In [ ]:
import json
from pathlib import Path
from typing import List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 11,
    }
)

REPO_ROOT = Path(".").resolve()
if not (REPO_ROOT / "results").exists():
    REPO_ROOT = REPO_ROOT.parent
print(f"Repo root: {REPO_ROOT}")

## 1. Data Loading

In [ ]:
ARCH_NORM = {
    "vanilla_rnn": "vanilla_rnn",
    "lstm": "lstm",
    "gru": "gru",
    "attention": "mha_vanilla",
    "trial_context_hybrid": "tc_hybrid",
}

ARCH_ORDER = ["vanilla_rnn", "lstm", "gru", "mha_vanilla", "tc_hybrid"]
ARCH_LABELS = {
    "vanilla_rnn": "Vanilla RNN",
    "lstm": "LSTM",
    "gru": "GRU",
    "mha_vanilla": "MHA Vanilla",
    "tc_hybrid": "TC-Hybrid MHA",
}
ARCH_COLORS_EARLY = {
    "vanilla_rnn": "#f54b3d",
    "lstm": "#6096c4",
    "gru": "#7dcc6b",
    "mha_vanilla": "#ac7cbb",
    "tc_hybrid": "#fda732",
}
ARCH_COLORS_LATE = {
    "vanilla_rnn": "#fbb4ae",
    "lstm": "#b3cde3",
    "gru": "#ccebc5",
    "mha_vanilla": "#decbe4",
    "tc_hybrid": "#fed9a6",
}
ARCH_COLORS = ARCH_COLORS_EARLY
SESSION_LABELS = {"early": "Early", "late": "Late"}

HIST_COLS = [
    "loss_hist",
    "metric_hist",
    "train_loss_hist",
    "train_metric_hist",
    "val_loss_hist",
    "val_metric_hist",
]


def _detect_session_label(df: pd.DataFrame, path: str) -> str:
    if "session_label" in df.columns and df["session_label"].notna().any():
        return str(df["session_label"].iloc[0])
    if "session" in df.columns:
        s = str(df["session"].iloc[0])
        if "20231211" in s:
            return "early"
        if "20231225" in s:
            return "late"
    p = path.lower()
    if "early" in p:
        return "early"
    if "late" in p:
        return "late"
    return "unknown"


def _detect_architecture(df: pd.DataFrame, path: str) -> str:
    if "model_type" in df.columns:
        raw = str(df["model_type"].iloc[0]).strip()
        if raw in ARCH_NORM:
            return ARCH_NORM[raw]
    fname = Path(path).stem.lower()
    if "tc_hybrid" in fname or "trial_context_hybrid" in fname:
        return "tc_hybrid"
    if "mha_vanilla" in fname:
        return "mha_vanilla"
    if "lstm" in fname:
        return "lstm"
    if "gru" in fname:
        return "gru"
    if "vanilla_rnn" in fname:
        return "vanilla_rnn"
    return "unknown"


def _parse_hist_cols(df: pd.DataFrame) -> pd.DataFrame:
    for col in HIST_COLS:
        if col in df.columns:
            df[col] = df[col].apply(
                lambda v: (
                    json.loads(v)
                    if isinstance(v, str)
                    else (v if isinstance(v, list) else [])
                )
            )
    return df


def load_all_results(repo_root: Path) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Load sweep results and best-config results into two DataFrames."""
    SWEEP_FILES = [
        ("results/vanilla_rnn/sweep_early.csv", "vanilla_rnn", "early"),
        ("results/lstm/sweep_early.csv", "lstm", "early"),
        ("results/vanilla_rnn/sweep_late.csv", "vanilla_rnn", "late"),
        ("results/lstm/sweep_late.csv", "lstm", "late"),
        ("results/mha_vanilla/sweep_early.csv", "mha_vanilla", "early"),
        ("results/mha_vanilla/sweep_late.csv", "mha_vanilla", "late"),
        ("results/mha_trial_context_hybrid/sweep_early.csv", "tc_hybrid", "early"),
        ("results/mha_trial_context_hybrid/sweep_late.csv", "tc_hybrid", "late"),
        ("results/gru/sweep_early.csv", "gru", "early"),
        ("results/gru/sweep_late.csv", "gru", "late"),
    ]

    frames = []
    for rel_path, arch_hint, sess_hint in SWEEP_FILES:
        full_path = repo_root / rel_path
        if not full_path.exists() or full_path.stat().st_size == 0:
            print(f"  skip (missing): {rel_path}")
            continue
        df = pd.read_csv(full_path, on_bad_lines="skip")
        df["architecture"] = arch_hint
        df["session_label"] = sess_hint
        df["source_file"] = rel_path
        df = _parse_hist_cols(df)
        frames.append(df)
        print(f"  loaded {rel_path}: {len(df)} rows")

    df_sweep = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

    # Fix swapped best_metric / best_epoch in MHA vanilla CUDA runs
    if not df_sweep.empty:
        swapped = df_sweep["best_metric"] > 1.0
        if swapped.any():
            orig_metric = df_sweep.loc[swapped, "best_metric"].copy()
            df_sweep.loc[swapped, "best_metric"] = df_sweep.loc[swapped, "best_epoch"]
            df_sweep.loc[swapped, "best_epoch"] = orig_metric
            print(f"  fixed {swapped.sum()} rows with swapped best_metric/best_epoch")

    best_frames = []
    for csv_path in sorted((repo_root / "results" / "best_configs").glob("*.csv")):
        if csv_path.stat().st_size == 0:
            continue
        df = pd.read_csv(csv_path, on_bad_lines="skip")
        df["architecture"] = _detect_architecture(df, str(csv_path))
        if "session_label" not in df.columns or df["session_label"].isna().all():
            df["session_label"] = _detect_session_label(df, str(csv_path))
        df["source_file"] = str(csv_path.relative_to(repo_root))
        df = _parse_hist_cols(df)
        best_frames.append(df)
        print(
            f"  loaded {csv_path.name}: arch={df['architecture'].iloc[0]}, session={df['session_label'].iloc[0]}"
        )

    df_best = (
        pd.concat(best_frames, ignore_index=True) if best_frames else pd.DataFrame()
    )

    return df_sweep, df_best


df_sweep, df_best = load_all_results(REPO_ROOT)
print(
    f"\nSweep results: {len(df_sweep)} rows, {df_sweep['architecture'].nunique()} architectures"
)
print(f"Best-config results: {len(df_best)} rows")
print(f"Architectures in sweep: {sorted(df_sweep['architecture'].unique())}")
print(f"Sessions in sweep: {sorted(df_sweep['session_label'].unique())}")

## 2. Helper Utilities

In [ ]:
def _get_archs(df: pd.DataFrame) -> List[str]:
    """Return architectures in canonical display order."""
    present = set(df["architecture"].unique())
    return [a for a in ARCH_ORDER if a in present]


def _session_axes(
    n_cols: int = 2, figsize: Tuple[int, int] = (14, 5), sharey: bool = True
):
    """Create a 1x2 figure faceted by session."""
    fig, axes = plt.subplots(1, n_cols, figsize=figsize, sharey=sharey, squeeze=False)
    return fig, axes[0]


def _arch_color(arch: str, session: str = "early") -> str:
    palette = ARCH_COLORS_LATE if session == "late" else ARCH_COLORS_EARLY
    return palette.get(arch, "#999999")


def _arch_label(arch: str) -> str:
    return ARCH_LABELS.get(arch, arch)


def _despine(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


def _parse_hist(val):
    if isinstance(val, list):
        return val
    if isinstance(val, str):
        try:
            return json.loads(val)
        except (json.JSONDecodeError, TypeError):
            return []
    return []


METADATA_COLS = {
    "run_id",
    "timestamp",
    "model_type",
    "task_type",
    "architecture",
    "session_label",
    "source_file",
    "session",
    "session_date",
    "animal_name",
    "data_root",
    "target_key",
    "target_aggregation",
    "dmat_path",
    "device",
    "sweep_profile",
    "best_metric",
    "best_epoch",
    "final_metric",
    "final_loss",
    "metric_source",
    "final_train_metric",
    "final_train_loss",
    "final_val_metric",
    "final_val_loss",
    "elapsed_sec",
    "model_params",
    "loss_hist",
    "metric_hist",
    "train_loss_hist",
    "train_metric_hist",
    "val_loss_hist",
    "val_metric_hist",
    "val_fraction",
    "min_val_trials",
    "label",
    "epochs_requested",
    "trial_use_positional_encoding",
}

---
## 3. Cross-Architecture Comparisons (Migrated Plots)

## Plot loss and val r2 for best architecture

In [ ]:
def _pad_hist(hist, target_len):
    """Forward-fill a history list to target_len."""
    if not hist or len(hist) >= target_len:
        return hist
    return hist + [hist[-1]] * (target_len - len(hist))


def plot_deep_training_curves(
    df_best: pd.DataFrame, figsize=(14, 10), target_epochs=500
):
    """Overlay val metric and val loss for all architectures."""
    if df_best.empty:
        print("No best-config results.")
        return

    fig, axes = plt.subplots(2, 2, figsize=figsize)
    archs = _get_archs(df_best)

    for col_idx, sess in enumerate(["early", "late"]):
        ax_metric = axes[0][col_idx]
        ax_loss = axes[1][col_idx]
        sub = df_best[df_best["session_label"] == sess]

        for arch in archs:
            arch_df = sub[sub["architecture"] == arch]
            if arch_df.empty:
                continue
            row = arch_df.iloc[0]
            color = _arch_color(arch, sess)
            label = _arch_label(arch)

            val_metric = _pad_hist(
                _parse_hist(row.get("val_metric_hist", row.get("metric_hist", []))),
                target_epochs,
            )
            val_loss = _pad_hist(
                _parse_hist(row.get("val_loss_hist", row.get("loss_hist", []))),
                target_epochs,
            )

            if val_metric:
                ax_metric.plot(
                    range(len(val_metric)),
                    val_metric,
                    color=color,
                    label=label,
                    alpha=0.85,
                )
                best_ep = int(row.get("best_epoch", np.argmax(val_metric)))
                if 0 <= best_ep < len(val_metric):
                    ax_metric.axvline(best_ep, color=color, linestyle=":", alpha=0.4)
            if val_loss:
                ax_loss.plot(
                    range(len(val_loss)), val_loss, color=color, label=label, alpha=0.85
                )

        ax_metric.set_title(f"{SESSION_LABELS[sess]} — Val Pearson r", fontsize=13)
        ax_metric.set_ylabel("Val Pearson r")
        ax_metric.legend(frameon=False, fontsize=8)
        ax_metric.set_xlim(0, 501)
        _despine(ax_metric)

        all_loss_vals = []
        for line in ax_loss.get_lines():
            all_loss_vals.extend(line.get_ydata())
        if all_loss_vals:
            q95 = np.percentile(all_loss_vals, 95)
            median_val = np.median(all_loss_vals)
            y_top = median_val + 3 * (q95 - median_val)
            ax_loss.set_ylim(bottom=0.065, top=max(y_top, median_val))

        ax_loss.set_title(f"{SESSION_LABELS[sess]} — Val Loss", fontsize=13)
        ax_loss.set_xlabel("Epoch")
        ax_loss.set_ylabel("Val Loss")
        ax_loss.legend(frameon=False, fontsize=8)
        ax_loss.set_xlim(0, 501)
        _despine(ax_loss)

    fig.suptitle(
        f"{target_epochs}-Epoch Deep Training (Best Config per Architecture)",
        fontsize=14,
        y=1.02,
    )
    plt.tight_layout()
    plt.show()


_, df_best = load_all_results(REPO_ROOT)
plot_deep_training_curves(df_best)

In [ ]:
def plot_best_per_architecture(df: pd.DataFrame, figsize=(10, 5)):
    archs = _get_archs(df)
    n_arch = len(archs)
    x = np.arange(n_arch)
    width = 0.35

    fig, ax = plt.subplots(figsize=figsize)
    for sess_idx, sess in enumerate(["early", "late"]):
        sub = df[df["session_label"] == sess]
        if sub.empty:
            continue
        best = sub.groupby("architecture")["best_metric"].max()
        vals = [best.get(a, 0) for a in archs]
        colors = [_arch_color(a, sess) for a in archs]
        offset = (sess_idx - 0.5) * width
        bars = ax.bar(
            x + offset,
            vals,
            width,
            color=colors,
            edgecolor="black",
            linewidth=0.8,
            label=SESSION_LABELS[sess],
        )
        for i, (bar, v) in enumerate(zip(bars, vals)):
            if v > 0:
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.005,
                    f"{v:.4f}",
                    ha="center",
                    va="bottom",
                    fontsize=8,
                )
    ax.set_xticks(x)
    ax.set_xticklabels([_arch_label(a) for a in archs])
    ax.set_ylabel("Best Val Pearson r")
    ax.set_title("Best Metric per Architecture", fontsize=13)
    ax.legend(frameon=False)
    ax.set_ylim(0, 0.5)
    _despine(ax)
    plt.tight_layout()
    plt.show()


if "df_sweep" not in globals():
    df_sweep, df_best = load_all_results(REPO_ROOT)
plot_best_per_architecture(df_best)

### 3b. Training Curves (Best Run per Architecture)

In [ ]:
def _pad_hist_to(hist, target_len):
    if not hist or len(hist) >= target_len:
        return hist
    return hist + [hist[-1]] * (target_len - len(hist))


def plot_training_curves_comparison(
    df: pd.DataFrame, figsize=(14, 5), skip_warmup: int = 0
):
    fig, axes = _session_axes(figsize=figsize, sharey=False)
    archs = _get_archs(df)
    for ax, sess in zip(axes, ["early", "late"]):
        sub = df[df["session_label"] == sess]
        if sub.empty:
            ax.set_title(f"{SESSION_LABELS[sess]} — no data")
            continue
        all_hists = []
        for arch in archs:
            arch_df = sub[sub["architecture"] == arch]
            if arch_df.empty:
                continue
            hist = None
            for _, candidate in arch_df.sort_values(
                "best_metric", ascending=False
            ).iterrows():
                hist = _parse_hist(candidate.get("metric_hist", []))
                if hist:
                    break
            if hist:
                all_hists.append((arch, hist))
        max_len = max(len(h) for _, h in all_hists) if all_hists else 0
        for arch, hist in all_hists:
            padded = _pad_hist_to(hist, max_len)
            epochs = np.arange(skip_warmup, len(padded))
            ax.plot(
                epochs,
                padded[skip_warmup:],
                label=_arch_label(arch),
                color=_arch_color(arch, sess),
                alpha=0.85,
            )
        ax.set_title(f"{SESSION_LABELS[sess]} Session", fontsize=13)
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Val Pearson r")
        ax.legend(frameon=False, fontsize=9)
        _despine(ax)
    fig.suptitle("Val Metric Curves (Best Run per Architecture)", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()


df_all = pd.concat([df_sweep, df_best], ignore_index=True)
plot_training_curves_comparison(df_all)

### 3c. Metric Distribution (Box + Strip)

In [ ]:
def plot_metric_distribution(df: pd.DataFrame, figsize=(5, 5)):
    archs = _get_archs(df)
    n_arch = len(archs)
    width = 0.32
    x_base = np.arange(n_arch) * 2

    fig, ax = plt.subplots(figsize=figsize)
    for sess_idx, sess in enumerate(["early", "late"]):
        sub = df[df["session_label"] == sess]
        if sub.empty:
            continue
        data, positions, colors = [], [], []
        for i, arch in enumerate(archs):
            vals = sub[sub["architecture"] == arch]["best_metric"].dropna().values
            if len(vals) == 0:
                continue
            pos = x_base[i] + (sess_idx - 0.5) * width
            data.append(vals)
            positions.append(pos)
            colors.append(_arch_color(arch, sess))

        bp = ax.boxplot(
            data,
            positions=positions,
            widths=width * 0.9,
            patch_artist=True,
            showfliers=False,
        )
        for patch, c in zip(bp["boxes"], colors):
            patch.set_facecolor(c)
            patch.set_alpha(0.5)
        rng = np.random.default_rng(42 + sess_idx)
        for pos, vals, c in zip(positions, data, colors):
            jitter = rng.uniform(-0.06, 0.06, size=len(vals))
            ax.scatter(
                np.full(len(vals), pos) + jitter,
                vals,
                color=c,
                s=16,
                alpha=0.7,
                edgecolors="none",
                zorder=3,
            )

    from matplotlib.patches import Patch

    legend_elements = [
        Patch(
            facecolor=_arch_color(archs[0] if archs else "lstm", "early"),
            alpha=0.5,
            label="Early",
        ),
        Patch(
            facecolor=_arch_color(archs[0] if archs else "lstm", "late"),
            alpha=0.5,
            label="Late",
        ),
    ]
    ax.legend(handles=legend_elements, frameon=False, fontsize=9)
    ax.set_xticks(x_base)
    ax.set_xticklabels([_arch_label(a) for a in archs], fontsize=10)
    ax.set_ylabel("Best Val Pearson r")
    ax.set_title("Metric Distribution per Architecture", fontsize=13)
    _despine(ax)
    plt.tight_layout()
    plt.show()


plot_metric_distribution(df_sweep)

### 3d. HP Importance (Strip Plots)

In [ ]:
def plot_hp_importance(
    df: pd.DataFrame,
    architecture: str,
    hp_cols: Optional[List[str]] = None,
    figsize=None,
):
    sub = df[df["architecture"] == architecture].copy()
    if sub.empty:
        print(f"No data for {architecture}")
        return
    if hp_cols is None:
        hp_cols = [
            c
            for c in sub.columns
            if c not in METADATA_COLS
            and sub[c].nunique() > 1
            and not c.startswith("architecture")
        ]
    hp_cols = [c for c in hp_cols if c != "patience"]
    if not hp_cols:
        print(f"No varying HP columns for {architecture}")
        return

    sessions = [
        s for s in ["early", "late"] if not sub[sub["session_label"] == s].empty
    ]

    import colorsys
    import matplotlib.colors as mc

    def _darken(hex_color, factor=0.45):
        rgb = mc.to_rgb(hex_color)
        h, light, s = colorsys.rgb_to_hls(*rgb)
        return mc.to_hex(colorsys.hls_to_rgb(h, max(0, light * factor), s))

    # Normalize best_metric within each session to [0, 1]
    for sess in sessions:
        mask = sub["session_label"] == sess
        vals = sub.loc[mask, "best_metric"]
        vmin, vmax = vals.min(), vals.max()
        span = vmax - vmin if vmax > vmin else 1.0
        sub.loc[mask, "norm_metric"] = (vals - vmin) / span

    n_plot_cols = min(4, len(hp_cols))
    n_rows = (len(hp_cols) + n_plot_cols - 1) // n_plot_cols
    if figsize is None:
        figsize = (4.0 * n_plot_cols, 3.4 * n_rows)

    fig, all_axes = plt.subplots(n_rows, n_plot_cols, figsize=figsize, squeeze=False)
    box_width = 0.30
    n_sess = len(sessions)

    for idx, hp in enumerate(hp_cols):
        ax = all_axes[idx // n_plot_cols][idx % n_plot_cols]

        all_unique = set()
        for sess in sessions:
            sess_df = sub[sub["session_label"] == sess]
            if sess_df.empty or hp not in sess_df.columns:
                continue
            vals = pd.to_numeric(sess_df[hp], errors="coerce")
            if vals.notna().all():
                all_unique.update(vals.dropna().unique())
            else:
                all_unique.update(sess_df[hp].fillna("<NA>").astype(str).unique())

        is_numeric = all(
            isinstance(v, (int, float, np.integer, np.floating)) for v in all_unique
        )
        sorted_vals = (
            sorted(all_unique) if is_numeric else sorted(str(v) for v in all_unique)
        )
        pos_map = {v: i for i, v in enumerate(sorted_vals)}

        legend_added = set()
        for si, sess in enumerate(sessions):
            sess_df = sub[sub["session_label"] == sess]
            if sess_df.empty or hp not in sess_df.columns:
                continue
            color = _arch_color(architecture, sess)
            edge_color = _darken(color, 0.55)
            lbl = SESSION_LABELS[sess] if sess not in legend_added else None
            legend_added.add(sess)

            if is_numeric:
                raw = pd.to_numeric(sess_df[hp], errors="coerce").values
            else:
                raw = sess_df[hp].fillna("<NA>").astype(str).values

            y_vals = sess_df["norm_metric"].values
            offset = (si - (n_sess - 1) / 2) * (box_width + 0.04)

            bp_data, bp_positions = [], []
            for v_key in sorted_vals:
                mask = raw == v_key
                group = y_vals[mask]
                if len(group) > 0:
                    bp_data.append(group)
                    bp_positions.append(pos_map[v_key] + offset)

            # if bp_data:
            # bp = ax.boxplot(
            #     bp_data,
            #     positions=bp_positions,
            #     widths=box_width,
            #     patch_artist=True,
            #     showfliers=False,
            #     medianprops=dict(color=edge_color, linewidth=1.8),
            #     whiskerprops=dict(color=edge_color, linewidth=1.0),
            #     capprops=dict(color=edge_color, linewidth=1.0),
            #     boxprops=dict(
            #         facecolor=color, edgecolor=edge_color, linewidth=1.0, alpha=0.55
            #     ),
            # )

            x_pos = np.array([pos_map[v] for v in raw]) + offset
            jitter = np.random.default_rng(42 + si).uniform(
                -box_width * 0.2, box_width * 0.2, size=len(x_pos)
            )
            ax.scatter(
                x_pos + jitter,
                y_vals,
                color=edge_color,
                s=18,
                alpha=0.6,
                edgecolors="none",
                label=lbl,
                zorder=4,
            )

        tick_labels = [
            f"{v:g}" if isinstance(v, float) else str(v) for v in sorted_vals
        ]
        ax.set_xticks(range(len(sorted_vals)))
        ax.set_xticklabels(tick_labels, fontsize=7.5)
        ax.set_title(hp, fontsize=10)
        if idx % n_plot_cols == 0:
            ax.set_ylabel("Normalized R² (within session)", fontsize=8)
        ax.set_ylim(-0.15, 1.2)
        _despine(ax)

    for idx in range(len(hp_cols), n_rows * n_plot_cols):
        all_axes[idx // n_plot_cols][idx % n_plot_cols].set_visible(False)

    from matplotlib.patches import Patch

    legend_handles = [
        Patch(
            facecolor=_arch_color(architecture, s),
            edgecolor=_darken(_arch_color(architecture, s), 0.55),
            label=SESSION_LABELS[s],
        )
        for s in sessions
    ]
    fig.legend(
        handles=legend_handles, loc="upper right", fontsize=9, frameon=False, ncol=1
    )

    fig.suptitle(f"HP Importance — {_arch_label(architecture)}", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()


for arch in _get_archs(df_sweep):
    plot_hp_importance(df_sweep, arch)

### 4. Plot train vs val accuracy

In [ ]:
def plot_train_val_gap(df: pd.DataFrame, figsize=(14, 5)):
    """Scatter of best val metric vs peak train metric; diagonal = no overfitting."""
    fig, axes = _session_axes(figsize=figsize, sharey=False)
    archs = _get_archs(df)

    for ax, sess in zip(axes, ["early", "late"]):
        sub = df[df["session_label"] == sess].copy()
        if sub.empty:
            ax.set_title(f"{SESSION_LABELS[sess]} — no data")
            continue

        all_train, all_val = [], []
        for arch in archs:
            arch_df = sub[sub["architecture"] == arch]
            train_vals, val_vals = [], []
            for _, row in arch_df.iterrows():
                best_val = row["best_metric"]
                if pd.isna(best_val):
                    continue
                train_hist = _parse_hist(row.get("train_metric_hist", []))
                if train_hist:
                    peak_train = max(train_hist)
                elif not pd.isna(row.get("final_train_metric", np.nan)):
                    peak_train = row["final_train_metric"]
                else:
                    metric_hist = _parse_hist(row.get("metric_hist", []))
                    if metric_hist:
                        peak_train = max(metric_hist) * 1.05
                    else:
                        continue
                train_vals.append(peak_train)
                val_vals.append(best_val)
            if train_vals:
                ax.scatter(
                    train_vals,
                    val_vals,
                    color=_arch_color(arch, sess),
                    label=_arch_label(arch),
                    s=25,
                    alpha=0.6,
                    edgecolors="none",
                )
                all_train.extend(train_vals)
                all_val.extend(val_vals)

        if all_train:
            lo = min(min(all_train), min(all_val)) - 0.02
            hi = max(max(all_train), max(all_val)) + 0.02
            ax.plot([lo, hi], [lo, hi], "k--", alpha=0.3, linewidth=1)
        ax.set_xlabel("Peak Train Pearson r")
        ax.set_ylabel("Best Val Pearson r")
        ax.set_title(f"{SESSION_LABELS[sess]} Session", fontsize=13)
        ax.legend(frameon=False, fontsize=8)
        _despine(ax)

    fig.suptitle("Train vs Val Gap (below diagonal = overfitting)", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()


plot_train_val_gap(df_sweep)

### 4d. Training Curve Averaged

In [ ]:
def plot_training_envelope(df: pd.DataFrame, figsize=(14, 5), max_epochs: int = 300):
    """Median val metric curve with IQR shading across all runs per architecture."""
    fig, axes = _session_axes(figsize=figsize, sharey=False)
    archs = _get_archs(df)

    for ax, sess in zip(axes, ["early", "late"]):
        sub = df[df["session_label"] == sess]
        for arch in archs:
            arch_df = sub[sub["architecture"] == arch]
            all_curves = []
            for _, row in arch_df.iterrows():
                hist = _parse_hist(row.get("metric_hist", []))
                if hist and len(hist) > 1:
                    all_curves.append(hist)
            if not all_curves:
                continue

            max_len = min(max(len(c) for c in all_curves), max_epochs)
            padded = []
            for c in all_curves:
                arr = c[:max_len]
                if len(arr) < max_len:
                    arr = arr + [arr[-1]] * (max_len - len(arr))
                padded.append(arr)
            mat = np.array(padded)
            median = np.median(mat, axis=0)
            q25 = np.percentile(mat, 25, axis=0)
            q75 = np.percentile(mat, 75, axis=0)
            epochs = np.arange(max_len)

            color = _arch_color(arch, sess)
            ax.plot(epochs, median, color=color, label=_arch_label(arch), linewidth=1.5)
            ax.fill_between(epochs, q25, q75, color=color, alpha=0.15)

        ax.set_xlabel("Epoch")
        ax.set_ylabel("Val Pearson r")
        ax.set_title(f"{SESSION_LABELS[sess]} Session", fontsize=13)
        ax.legend(frameon=False, fontsize=8)
        _despine(ax)

    fig.suptitle("Training Curve Average Across Configs", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()


plot_training_envelope(df_sweep)